In [9]:
import time
import numpy as np
 
def extract_features(seed):
    rng = np.random.default_rng(seed)
    data = rng.standard_normal((800, 800))
    _ = np.linalg.svd(data, full_matrices=False)
    # time.sleep(0.25)
    return True
 
start = time.perf_counter()
for i in range(908):
    extract_features(i)
print(f"Time: {time.perf_counter() - start:.2f}s")
# Output: ~10 seconds

Time: 45.44s


In [8]:
import time
import numpy as np
import ray
 
@ray.remote
def extract_features(seed):
    rng = np.random.default_rng(seed)
    data = rng.standard_normal((800, 800))
    _ = np.linalg.svd(data, full_matrices=False)
    # time.sleep(0.25)
    return True
 
ray.init()
start = time.perf_counter()
futures = [extract_features.remote(i) for i in range(9008)]
ray.get(futures)
print(f"Time: {time.perf_counter() - start:.2f}s")
ray.shutdown()


2026-03-18 13:08:18,970	INFO worker.py:2013 -- Started a local Ray instance.


Time: 90.36s


In [10]:
import ray
ray.init()

2026-03-19 11:44:05,086	INFO worker.py:2013 -- Started a local Ray instance.


Python version:,3.13.11
Ray version:,2.54.0


In [11]:
print(ray.cluster_resources())


{'CPU': 14.0, 'memory': 25164775424.0, 'node:127.0.0.1': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 2147483648.0}


In [20]:
import ray
ray.init()
 
# Open http://localhost:8265
 
@ray.remote
def debug_task():
    import time, socket
    time.sleep(30)
    return f"Executed on {socket.gethostname()}"
 
refs = [debug_task.remote() for _ in range(50)]
results = ray.get(refs)
print(results)

2026-03-19 12:11:43,848	INFO worker.py:2013 -- Started a local Ray instance.


['Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on MES-ASS-0191-KULDEEPS-MAC.local', 'Executed on

In [42]:
import ray
import time
import socket
import os

ray.init()

@ray.remote
def debug_task(task_id):
    """Diagnostic task that reveals Ray's internals."""
    start = time.time()
    time.sleep(2)  # Simulate work
    
    return {
        "task_id": task_id,
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "duration": round(time.time() - start, 2),
    }

# Launch more tasks than you have CPUs to see queuing behavior
num_tasks = 500
print(f"Submitting {num_tasks} tasks...")

refs = [debug_task.remote(i) for i in range(num_tasks)]

# Retrieve results as they complete (not all at once)
unfinished = refs.copy()
while unfinished:
    finished, unfinished = ray.wait(unfinished, num_returns=1)
    result = ray.get(finished[0])
    print(f"  Task {result['task_id']} done on PID {result['pid']} "
          f"in {result['duration']}s")

# Key observations:
# - You'll see ~CPU_COUNT tasks finish in the first wave (~2s)
# - Then the next wave, and so on
# - Different PIDs = different workers
# - The Raylet queued excess tasks until workers freed up

2026-03-19 12:51:36,335	INFO worker.py:2013 -- Started a local Ray instance.


Submitting 500 tasks...
  Task 2 done on PID 92600 in 2.0s
  Task 5 done on PID 92595 in 2.0s
  Task 4 done on PID 92596 in 2.0s
  Task 3 done on PID 92597 in 2.0s
  Task 0 done on PID 92598 in 2.0s
  Task 6 done on PID 92599 in 2.0s
  Task 7 done on PID 92603 in 2.0s
  Task 1 done on PID 92593 in 2.0s
  Task 8 done on PID 92601 in 2.0s
  Task 9 done on PID 92594 in 2.0s
  Task 10 done on PID 92602 in 2.0s
  Task 11 done on PID 92604 in 2.0s
  Task 12 done on PID 92605 in 2.0s
  Task 13 done on PID 92606 in 2.0s
  Task 14 done on PID 92600 in 2.0s
  Task 15 done on PID 92595 in 2.0s
  Task 17 done on PID 92598 in 2.0s
  Task 16 done on PID 92601 in 2.0s
  Task 18 done on PID 92593 in 2.0s
  Task 19 done on PID 92599 in 2.0s
  Task 21 done on PID 92597 in 2.0s
  Task 20 done on PID 92596 in 2.0s
  Task 22 done on PID 92603 in 2.0s
  Task 23 done on PID 92594 in 2.0s
  Task 24 done on PID 92602 in 2.0s
  Task 25 done on PID 92604 in 2.0s
  Task 26 done on PID 92605 in 2.0s
  Task 27 done

In [41]:
ray.shutdown()